# 03. Config/Seed/Submission Ledger? Inference ??? ??

> ???: 2026-07-07 KST  
> ??: baseline RandomForest inference? ?? ??, seed, commit, data manifest hash, submission SHA256, ledger row? ????.  
> ??: ? ???? **?? ?? CSV, ?? ??, outputs ???**? ???? ???. ??? ??? ??? synthetic smoke? ????, ?? ??? ???.

## ?? ??

1. ?? `config + seed + code commit + data manifest`? baseline inference? ?? ???? ?? submission hash? ?????
2. ledger row?? 2? ??? ?? ?? ??? ??? ?? ??? ????
3. hash? ??? CSV bytes? ?? ??? ??? CSV bytes? ?? canonical serializer? ?????
4. notebook? ?? ??? ? ??? ?? ?? ?????? ??? ? ????


## Executive Summary

?? ??? ?? ?? ??? ??? **?? ?? ???**? ??? ?? ???? ??? ????. Dacon 2? ????? train/inference ??? ??? Private score? ?? ?? ??? ?? ?? ? ??? ??. ??? ?? ??? ?? CSV? ??? ?? ??? ?? ?? ?? ??? ?????? ??.

| ???? | ?? ?? | ?? ???? ?? ?? |
|---|---|---|
| `BaselineRunConfig` | ?? ID, seed, ?? ????? ? ?? ??? ?? | stable hash? seed ?? ?? |
| seed | RandomForest? numpy/Python random? ?? ?? | ?? seed ?? ?? hash ??? ?? |
| commit/data manifest | ????? ??? ?? ??? | ledger row ??? ?? |
| submission SHA256 | ?? ?? ?? ??? byte-level ?? | canonical CSV bytes? hash ?? |
| submission ledger | ?? ??, score, ?? ???? ?? ?? ? | ? ? DataFrame schema ?? |

**Decision Box**

| ?? | ?? | ?? |
|---|---|---|
| ?? ???? synthetic smoke? ?? | ?? ?? ??? RandomForest ??? ??? ?? ?? CSV? ?? ??? ??, ??? ?? ??? ?? | ?? |
| ?? ?? ??? ?? ?? | ?? CSV? outputs? Git ?? ??? ???, ?? ??? ?? CLI ???? ?? ?? ?? | ?? |
| hash? `submission_to_csv_bytes()` ???? ?? | ledger hash? ?? ?? ?? bytes? ???? ???? ?? | ?? |


In [1]:
from pathlib import Path
import hashlib
import os
import subprocess
import sys
import tempfile

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
  sys.path.insert(0, str(SRC_DIR))

from baram.metrics import TARGET_COLS
from baram.reproducibility import (
  SUBMISSION_LEDGER_COLUMNS,
  BaselineRunConfig,
  build_submission_filename,
  ledger_entry_to_frame,
  run_baseline_with_reproducibility,
  submission_csv_sha256,
  submission_to_csv_bytes,
  verify_baseline_inference_reproducibility,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TARGET_COLS:", TARGET_COLS)


PROJECT_ROOT: C:\Users\kik32\workspace\Dacon\2026-BARAM-Wind-Power-Prediction-AI-Competition
TARGET_COLS: ['kpx_group_1', 'kpx_group_2', 'kpx_group_3']


## Synthetic Smoke Fixture ??

?? synthetic ???? ?? ?? ???? ???? ?? ??? ???.

- `train_labels`: `kst_dtm`? target 3?. Group 3? ?? ??? ??? target? non-null mask? ???? ??.
- `ldaps/gfs`: `forecast_kst_dtm`, `data_available_kst_dtm`, grid metadata, u/v ?? ??? ???.
- `sample_submission`: `forecast_id`, `forecast_kst_dtm`, target 3?? ?? template ??? ??.

??? ?? ???? synthetic smoke? ??? ??? ???. ?? **?? ??**? ????. ? ??? ???? ???? ?? API? ?? schema? ?? hash ??? ??????? ?? ???.


In [2]:
def make_weather_frame(times, values):
  rows = []
  for time_index, forecast_time in enumerate(pd.to_datetime(times)):
    available_time = forecast_time - pd.Timedelta(hours=12)
    for grid_id in [1, 2]:
      value = float(values[time_index] + grid_id)
      rows.append(
        {
          "forecast_kst_dtm": forecast_time,
          "data_available_kst_dtm": available_time,
          "grid_id": grid_id,
          "latitude": 37.0 + grid_id * 0.01,
          "longitude": 128.0 + grid_id * 0.01,
          "heightAboveGround_10_10u": value,
          "heightAboveGround_10_10v": value * 2,
        }
      )
  return pd.DataFrame(rows)


def make_label_frame():
  times = pd.date_range("2024-01-01 01:00:00", periods=5, freq="h")
  return pd.DataFrame(
    {
      "kst_dtm": times,
      "kpx_group_1": [1000.0, 3000.0, 5000.0, 7000.0, 9000.0],
      "kpx_group_2": [2000.0, 4000.0, 6000.0, 8000.0, 10000.0],
      "kpx_group_3": [np.nan, np.nan, 2100.0, 5000.0, 8000.0],
    }
  )


def make_sample_submission(times):
  return pd.DataFrame(
    {
      "forecast_id": [f"F{i:03d}" for i in range(len(times))],
      "forecast_kst_dtm": pd.to_datetime(times),
      "kpx_group_1": 0.0,
      "kpx_group_2": 0.0,
      "kpx_group_3": 0.0,
    }
  )


def tiny_run_inputs():
  labels = make_label_frame()
  sample = make_sample_submission(pd.date_range("2025-01-01 01:00:00", periods=2, freq="h"))
  return {
    "train_labels": labels,
    "ldaps_train": make_weather_frame(labels["kst_dtm"], [1, 2, 3, 4, 5]),
    "gfs_train": make_weather_frame(labels["kst_dtm"], [10, 20, 30, 40, 50]),
    "sample_submission": sample,
    "ldaps_test": make_weather_frame(sample["forecast_kst_dtm"], [6, 7]),
    "gfs_test": make_weather_frame(sample["forecast_kst_dtm"], [60, 70]),
  }

inputs = tiny_run_inputs()
display(pd.DataFrame({
  "frame": list(inputs.keys()),
  "rows": [len(frame) for frame in inputs.values()],
  "columns": [len(frame.columns) for frame in inputs.values()],
}))


,frame,rows,columns
0,train_labels,5,4
1,ldaps_train,10,7
2,gfs_train,10,7
3,sample_submission,2,5
4,ldaps_test,4,7
5,gfs_test,4,7


### ??: smoke fixture? ???? ?

? fixture? 2??? ?? template? ??? ????, ?? baseline pipeline? ?? ???? ????.

1. weather aggregation? calendar feature ??
2. target? non-null label mask
3. RandomForest seed ??
4. sample submission? ID/time ??
5. capacity clip ?? submission assembly

??? ? ???? ??? ??? ???? ??? ???? ledger? baseline inference ??? ??? ????? ????.


In [3]:
config = BaselineRunConfig(
  experiment_id="rf_smoke",
  seed=7,
  model_params={"n_estimators": np.int64(4), "n_jobs": 1, "when": pd.Timestamp("2026-07-07 09:00:00", tz="Asia/Seoul"), "choices": {"b", "a"}},
  notes="synthetic reproducibility smoke",
)
config_native = BaselineRunConfig(
  experiment_id="rf_smoke",
  seed=7,
  model_params={"choices": ["a", "b"], "n_estimators": 4, "n_jobs": 1, "when": "2026-07-07T09:00:00+09:00"},
  notes="synthetic reproducibility smoke",
)

config_audit = pd.DataFrame([
  {"check": "random_state is forced by seed", "value": config.effective_model_params()["random_state"], "pass": config.effective_model_params()["random_state"] == 7},
  {"check": "stable hash ignores param order/native wrappers", "value": config.config_sha256()[:12], "pass": config.config_sha256() == config_native.config_sha256()},
  {"check": "seed change changes config hash", "value": BaselineRunConfig("rf_smoke", seed=8, model_params={"n_estimators": 4}).config_sha256()[:12], "pass": config.config_sha256() != BaselineRunConfig("rf_smoke", seed=8, model_params={"n_estimators": 4}).config_sha256()},
])
display(config_audit)


,check,value,pass
0,random_state is forced by seed,7,True
1,stable hash ignores param order/native wrappers,989671c4423b,True
2,seed change changes config hash,40b56fd867f1,True


### ??: config hash? ??

`config_sha256`? ?? ??? ????. ??? ??? ?? ? ???.

- `random_state`? ???? ?? ??? ?? config??? `seed`? ????. seed? ledger? ?? ?? ???? ??.
- numpy scalar, pandas timestamp, set?? ?? ???? ?? ??? ??? stable JSON?? ?????. ? ??? ??? ?? ??? ??? ?? ??? ?? hash ?? ?? hash ???? ?? ? ??.

?? ??: config hash? ??? ?? ?? ? ?? ?? ???? ???? 1? ??? ??? ? ??.


In [4]:
filename = build_submission_filename("exp/a b", "abcdef123456", "2026-07-07T09:00:00+09:00")
submission = inputs["sample_submission"].copy()
canonical_bytes = submission_to_csv_bytes(submission)
sha_from_bytes = hashlib.sha256(canonical_bytes).hexdigest()
sha_from_helper = submission_csv_sha256(submission)

serializer_audit = pd.DataFrame([
  {"check": "filename is path-safe", "value": filename, "pass": filename == "submission_20260707_exp-a-b_abcdef1.csv"},
  {"check": "hash uses canonical bytes", "value": sha_from_helper[:12], "pass": sha_from_bytes == sha_from_helper},
  {"check": "canonical bytes include UTF-8 BOM", "value": canonical_bytes[:3], "pass": canonical_bytes.startswith(b"\xef\xbb\xbf")},
])
display(serializer_audit)


,check,value,pass
0,filename is path-safe,submission_20260707_exp-a-b_abcdef1.csv,True
1,hash uses canonical bytes,8d42fe36c808,True
2,canonical bytes include UTF-8 BOM,b'\xef\xbb\xbf',True


### ??: filename? CSV bytes? ?? ??? ??

?? ???? ?? ?? ??? ??? DataFrame?? ?? ??? ?? ?? ??? ??? ???. ?? ?? ???, BOM, ?? ???, float ??? ??? ??? SHA256? ????.

?? ??? `submission_to_csv_bytes()`? ?? serializer? ???. ?? ???? ?? ?? ?? ??? ?? ?? ? bytes? ??? ??, ledger hash? ?? ?? hash? ??? ???? ?? ? ??.

?? ??: ?? ????? ???? ?? hash? ????. ?? ??? submission validator CLI? ?? ??? ?? ????.


In [5]:
with tempfile.TemporaryDirectory() as temp_dir:
  before = sorted(os.listdir(temp_dir))
  old_cwd = Path.cwd()
  os.chdir(temp_dir)
  try:
    result = run_baseline_with_reproducibility(
      config=BaselineRunConfig(
        experiment_id="rf_smoke",
        seed=7,
        model_params={"n_estimators": 4, "n_jobs": 1},
        notes="notebook smoke",
      ),
      **inputs,
      git_commit="abcdef123456",
      data_manifest_sha256="manifest-sha",
      created_at_kst="2026-07-07T09:00:00+09:00",
    )
  finally:
    os.chdir(old_cwd)
  after = sorted(os.listdir(temp_dir))

ledger_entry = result["ledger_entry"]
ledger_frame = ledger_entry_to_frame(ledger_entry)

key_columns = [
  "created_at_kst",
  "experiment_id",
  "git_commit",
  "seed",
  "config_sha256",
  "data_manifest_sha256",
  "submission_filename",
  "submission_sha256",
  "submission_rows",
  "selected",
  "note",
]
display(ledger_frame[key_columns])
print("temporary directory before:", before)
print("temporary directory after :", after)
print("submission shape:", result["baseline_result"]["submission"].shape)


,created_at_kst,experiment_id,git_commit,seed,config_sha256,data_manifest_sha256,submission_filename,submission_sha256,submission_rows,selected,note
0,2026-07-07T09:00:00+09:00,rf_smoke,abcdef123456,7,a38005d7664e4e46b5c48c393dec279e7086c8a2bcc356...,manifest-sha,submission_20260707_rf_smoke_abcdef1.csv,3c3ab54d103ad3d4de79a150853b7a60a54a82bb49b76f...,2,False,notebook smoke


temporary directory before: []
temporary directory after : []
submission shape: (2, 5)


### ??: ledger row? ??? ????

? ledger row? ?? `outputs/submissions/submission_ledger.csv`? ??? ? ?? ????.

- `created_at_kst`: ?? ?? ?? ??? KST ???? ????.
- `experiment_id`: ?? ??? ????.
- `git_commit`: ??? ??? ????.
- `seed`? `config_sha256`: ?? ??? ??? ????.
- `data_manifest_sha256`: ?? ??? ?? ??? ????.
- `submission_filename`? `submission_sha256`: ?? ?? ??? byte-level ?? ????.

?? ???? before/after? ?? ?? ??? ? ??? ??? ?? ???? ???. ? ?? ??? ?? ??? ??? **ledger metadata ??? reproducibility check**?? ??? ??.


In [6]:
report = verify_baseline_inference_reproducibility(
  config=BaselineRunConfig(
    experiment_id="rf_smoke",
    seed=7,
    model_params={"n_estimators": 4, "n_jobs": 1},
    notes="notebook repeatability check",
  ),
  **inputs,
  git_commit="abcdef123456",
  data_manifest_sha256="manifest-sha",
  repeats=2,
  created_at_kst="2026-07-07T09:00:00+09:00",
)

run_frame = pd.DataFrame(report["runs"])
summary = pd.DataFrame([
  {"check": "two runs requested", "value": report["repeats"], "pass": report["repeats"] == 2},
  {"check": "same submission sha256", "value": run_frame["submission_sha256"].nunique(), "pass": report["is_reproducible"]},
  {"check": "same config sha256", "value": run_frame["config_sha256"].nunique(), "pass": run_frame["config_sha256"].nunique() == 1},
])
display(run_frame)
display(summary)


,run_index,created_at_kst,submission_filename,submission_sha256,config_sha256
0,1,2026-07-07T09:00:00+09:00,submission_20260707_rf_smoke_abcdef1.csv,3c3ab54d103ad3d4de79a150853b7a60a54a82bb49b76f...,fb6975e17b385f5021c9cffe77c33c59702aeceeea3f50...
1,2,2026-07-07T09:00:00+09:00,submission_20260707_rf_smoke_abcdef1.csv,3c3ab54d103ad3d4de79a150853b7a60a54a82bb49b76f...,fb6975e17b385f5021c9cffe77c33c59702aeceeea3f50...


,check,value,pass
0,two runs requested,2,True
1,same submission sha256,1,True
2,same config sha256,1,True


### ??: ?? inference ??

?? ??? `submission_sha256`? ??? ?? ?? ??? synthetic smoke ???? ????? ???.

1. seed? RandomForest? ????.
2. feature column order? imputer ??? ?? ?? ??? ???? ???.
3. submission assembly? CSV serialization? ?? bytes? ???.
4. ledger row? ?? fingerprint? ?? ?? ??? ????.

??? ????. ? ??? ?? ?? ???? ? ?? ??, OS/????? ?? ???? ???? ???. ?? ?? ????? ?? contract? ?? ???? ??? model artifact? ???? ??.


In [7]:
commands = [
  [sys.executable, "-m", "pytest", "tests/test_reproducibility.py", "-q"],
  [sys.executable, "-m", "pytest", "-q"],
]

for command in commands:
  completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
  print("$", " ".join(command))
  print(completed.stdout)
  if completed.stderr:
    print(completed.stderr)
  print("returncode:", completed.returncode)
  assert completed.returncode == 0


$ C:\Users\kik32\anaconda3\python.exe -m pytest tests/test_reproducibility.py -q
.........                                                                [100%]
9 passed in 4.56s

returncode: 0


$ C:\Users\kik32\anaconda3\python.exe -m pytest -q
....................                                                     [100%]
20 passed in 5.21s

returncode: 0


## Final Decision Box

| ?? | ?? | ?? | ?? ?? |
|---|---|---|---|
| Config/seed ?? | ?? | seed? `random_state`? ???? config hash? ??? | ?? config ?? ??? ?? |
| Submission hash | ?? | canonical bytes? SHA256 helper? ?? ?? ?? | ?? CLI??? ?? bytes ?? |
| Ledger row | ?? | ?? ?? ??? ? ? DataFrame ?? | `outputs/submissions/submission_ledger.csv` ?? ?? ?? |
| ?? inference | ?? | 2? ?? submission SHA256 ?? | ?? ??? smoke? ?? artifact ??? ?? ?? |
| ?? ?? ?? | ?? | ?? ???? before/after ?? ?? | ?? CLI ???? ?? ??? ????? ?? |

## ?? ?? ??

?? ??? `submission validator CLI? ?? train.py/inference.py ??? ??`?. ?, ?? ??? ?? ???? ??? ???? ???.

1. ?? notebook? validator ????? ?? ??? ?????? ????.
2. ? ?? `src/baram/validation.py` ?? `src/baram/submission.py`? validator? TDD? ????.
3. ????? `train.py`? `inference.py` CLI? ???, ?? CSV ??? validator ?? ??? ????.
4. ?? ??? `.ipynb`? ???, ??, ??, Decision Box? ???.
